In [ ]:
# Check Python environment /user/bin/env python3 -> running in Colab
import sys
print(sys.executable)

# Check Colab specific modules, True -> running in Colab, False -> not running in Colab
import sys
"google.colab" in sys.modules



In [7]:
# NOTE: 
#   - ** POC ONLY ** Optimizing RMSE against a fake priority score does not correlate with real KPIs.
#   - Labels should come from historical execution data

import pandas as pd # DataFrame and Series for structured data manipulation
import numpy as np # High performance numerical computing, arrays and vectorized ops
from sklearn.model_selection import train_test_split # Machine learning preprocessing, modeling and validation
from sklearn.ensemble import GradientBoostingRegressor # Forecasting method that sequentially trains decision trees to learn rules that minimize prediction loss.
from sklearn.metrics import root_mean_squared_error # Average magnitude of prediction error

# Generating synthetic task data
np.random.seed(48) # Seed for reproducible random sequence
num_tasks = 1100 # Number of tasks

# Create flattened table of task features
data = pd.DataFrame({
    "task_id": range(1, num_tasks + 1),
    "processing_time": np.random.uniform(1, 60, num_tasks),
    "deadline_minutes": np.random.uniform(5, 180, num_tasks),
    "urgency": np.random.randint(1, 6, num_tasks),  # 1 (low) to 5 (high)
    "resource_load": np.random.uniform(0.1, 1.0, num_tasks),
    "distance": np.random.uniform(1, 500, num_tasks)
})

# Display first 5 rows
# print(data.head(5))

# print(data.describe())       # stats for numeric columns
# print(data.info())           # overview of column types and nulls

# Define priority_score as target label
# Invert and normalize features where smaller values = higher priority
deadline_inv = 1 / data["deadline_minutes"]
deadline_inv_norm = deadline_inv / deadline_inv.max()  # normalize to 0-1

distance_inv = 1 / data["distance"]
distance_inv_norm = distance_inv / distance_inv.max()  # normalize to 0-1

# Normalize other features to 0-1
urgency_norm = data["urgency"] / data["urgency"].max()
processing_norm = data["processing_time"] / data["processing_time"].max()

# Compute weighted priority score using normalized features
data["priority_score"] = (
    0.5 * deadline_inv_norm +  # normalized inverse of deadline
    0.3 * urgency_norm -       # normalized urgency
    0.2 * processing_norm -    # normalized processing time
    0.1 * distance_inv_norm    # normalized inverse of distance
)

# Optional: scale final score to 0-1 for easy interpretation
data["priority_score_scaled"] = (
    (data["priority_score"] - data["priority_score"].min()) /
    (data["priority_score"].max() - data["priority_score"].min())
)

# print(data.head(5))

# Split data into features and target label
features_X = data[["processing_time", "deadline_minutes", "urgency", "resource_load", "distance"]]
target_y = data["priority_score"]

X_train, X_test, y_train, y_test = train_test_split(features_X, target_y, test_size=0.2, random_state=42)

# Train model
model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate, predict against input
predictions = model.predict(X_test)
rmse = root_mean_squared_error(y_test, predictions)

print(f"RMSE: {rmse:.4f}")

# Testing batch inference
incoming_tasks = pd.DataFrame([
    {"processing_time": 10, "deadline_minutes": 30, "urgency": 5, "resource_load": 0.4, "distance": 120},
    {"processing_time": 40, "deadline_minutes": 90, "urgency": 2, "resource_load": 0.7, "distance": 300},
    {"processing_time": 15, "deadline_minutes": 45, "urgency": 4, "resource_load": 0.3, "distance": 80},
])

incoming_tasks["predicted_priority"] = model.predict(incoming_tasks)
sequenced_tasks = incoming_tasks.sort_values(
    by="predicted_priority",
    ascending=False
)

print("\nTask Execution Order:")
print(sequenced_tasks[:10])  # Display top 10 tasks in execution order

RMSE: 0.0119

Task Execution Order:
   processing_time  deadline_minutes  urgency  resource_load  distance  \
0               10                30        5            0.4       120   
2               15                45        4            0.3        80   
1               40                90        2            0.7       300   

   predicted_priority  
0            0.335815  
2            0.248790  
1            0.015275  


In [ ]:
import sklearn
print(sklearn.__version__)
